In [23]:
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [24]:
# Menginisialisasi factory untuk stopword & stemmer Sastrawi, serta mendefinisikan korpus teks
pabrik_stopword = StopWordRemoverFactory()
kumpulan_stopword = pabrik_stopword.get_stop_words()
pabrik_stemmer = StemmerFactory()
mesin_stem = pabrik_stemmer.create_stemmer()

korpus_teks = [
    "Pelatihan Kecerdasan Buatan (AI) akan diselenggarakan di Fakultas Teknik UNM pada awal tahun 2026 mendatang.",
    "Jaringan komputer di laboratorium sedang diperbaiki agar koneksi internet mahasiswa menjadi jauh lebih stabil.",
    "Penerapan sistem temu kembali informasi sangat bergantung pada kualitas algoritma pemrosesan bahasa alami.",
    "Bagi seluruh mahasiswa tingkat akhir, pengumpulan draf laporan skripsi wajib dilakukan paling lambat hari Jumat",
    "Penggunaan framework React dan arsitektur database modern sangat esensial bagi pengembang web profesional."
]

In [25]:
# Fungsi utama untuk menjalankan tahapan case folding, cleansing, filtering, dan stemming
def eksekusi_preprocessing(teks):
    teks_kecil = teks.lower()
    teks_filter = re.sub(r'[^a-z\s]', ' ', teks_kecil)
    token_kotor = teks_filter.split()
    token_terfilter = [kata for kata in token_kotor if kata not in kumpulan_stopword]
    teks_sementara = ' '.join(token_terfilter)
    teks_akar = mesin_stem.stem(teks_sementara)
    token_bersih = teks_akar.split()
    return token_kotor, token_bersih

In [26]:
# Iterasi memproses korpus teks dan menghitung statistik penyusutan (reduksi) token
koleksi_hasil = []
for indeks, kalimat in enumerate(korpus_teks):
    sebelum, sesudah = eksekusi_preprocessing(kalimat)
    jum_sebelum = len(sebelum)
    jum_sesudah = len(sesudah)
    persentase_turun = ((jum_sebelum - jum_sesudah) / jum_sebelum) * 100 if jum_sebelum > 0 else 0
    
    koleksi_hasil.append({
        "ID": f"Dokumen-{indeks+1}",
        "Teks Original": kalimat,
        "Token Mentah": sebelum,
        "Token Final": sesudah,
        "Total Awal": jum_sebelum,
        "Total Akhir": jum_sesudah,
        "Reduksi (%)": round(persentase_turun, 2)
    })

In [27]:
# Menampilkan komparasi hasil teks secara visual untuk dua dokumen pertama
print("======================================================")
print("   KOMPARASI TEKS: SEBELUM & SESUDAH PREPROCESSING    ")
print("======================================================")
for x in range(2):
    print(f"[{koleksi_hasil[x]['ID']}]")
    print(f"[-] Asli  : {koleksi_hasil[x]['Teks Original']}")
    print(f"[-] Sebelum  : {koleksi_hasil[x]['Token Mentah']}")
    print(f"[-] Setelah  : {koleksi_hasil[x]['Token Final']}\n")

   KOMPARASI TEKS: SEBELUM & SESUDAH PREPROCESSING    
[Dokumen-1]
[-] Asli  : Pelatihan Kecerdasan Buatan (AI) akan diselenggarakan di Fakultas Teknik UNM pada awal tahun 2026 mendatang.
[-] Sebelum  : ['pelatihan', 'kecerdasan', 'buatan', 'ai', 'akan', 'diselenggarakan', 'di', 'fakultas', 'teknik', 'unm', 'pada', 'awal', 'tahun', 'mendatang']
[-] Setelah  : ['latih', 'cerdas', 'buat', 'ai', 'selenggara', 'fakultas', 'teknik', 'unm', 'awal', 'tahun', 'datang']

[Dokumen-2]
[-] Asli  : Jaringan komputer di laboratorium sedang diperbaiki agar koneksi internet mahasiswa menjadi jauh lebih stabil.
[-] Sebelum  : ['jaringan', 'komputer', 'di', 'laboratorium', 'sedang', 'diperbaiki', 'agar', 'koneksi', 'internet', 'mahasiswa', 'menjadi', 'jauh', 'lebih', 'stabil']
[-] Setelah  : ['jaring', 'komputer', 'laboratorium', 'sedang', 'baik', 'koneksi', 'internet', 'mahasiswa', 'jadi', 'jauh', 'lebih', 'stabil']



In [28]:
# Menampilkan tabel rekapitulasi data penyusutan menggunakan Pandas DataFrame
tabel_statistik = pd.DataFrame(koleksi_hasil)
print("======================================================")
print("             REKAPITULASI PENYUSUTAN TOKEN            ")
print("======================================================")
display(tabel_statistik[['ID', 'Total Awal', 'Total Akhir', 'Reduksi (%)']])

             REKAPITULASI PENYUSUTAN TOKEN            


,ID,Total Awal,Total Akhir,Reduksi (%)
0,Dokumen-1,14,11,21.43
1,Dokumen-2,14,12,14.29
2,Dokumen-3,13,11,15.38
3,Dokumen-4,15,14,6.67
4,Dokumen-5,13,11,15.38


**Analisis Preprocessing:** Preprocessing menghilangkan noise berupa tanda baca, angka, dan kata sambung (stopwords) yang tidak memiliki nilai pencarian spesifik, serta menyatukan variasi imbuhan menjadi akar kata (stemming). Dampak utamanya terhadap kualitas data sistem Information Retrieval (IR) adalah penurunan dimensi vocabulary secara drastis yang membuat ukuran indeks menjadi jauh lebih kecil dan ringan. Ini tidak hanya mempercepat komputasi saat pencarian dokumen teknis atau berita, tetapi juga meningkatkan relevansi (recall) karena sistem akan mencocokkan akar makna kata pencari secara tepat tanpa terganggu oleh perbedaan struktur tata bahasa dokumen aslinya.